<a href="https://colab.research.google.com/github/ArkanUbaidillah/Arkan-Ubaidillah-Warman_2411537001_ML2526/blob/main/Praktikum5/TugasCrossValidation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [8]:
train_url = "https://raw.githubusercontent.com/ArkanUbaidillah/Arkan-Ubaidillah-Warman_2411537001_ML2526/main/Praktikum5/train.csv"
test_url = "https://raw.githubusercontent.com/ArkanUbaidillah/Arkan-Ubaidillah-Warman_2411537001_ML2526/main/Praktikum5/test.csv"
submit_url = "https://raw.githubusercontent.com/ArkanUbaidillah/Arkan-Ubaidillah-Warman_2411537001_ML2526/main/Praktikum5/gender_submission.csv"

train_df = pd.read_csv(train_url)
test_df = pd.read_csv(test_url)
submit_df = pd.read_csv(submit_url)

In [9]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

train = train_df[['Survived'] + features].copy()
test = test_df[features].copy()

In [10]:
imputer = SimpleImputer(strategy='mean')

train['Age'] = imputer.fit_transform(train[['Age']])
test['Age'] = imputer.transform(test[['Age']])

train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])
test['Embarked'] = test['Embarked'].fillna(train['Embarked'].mode()[0])

In [11]:
le = LabelEncoder()

train['Sex'] = le.fit_transform(train['Sex'])
test['Sex'] = le.transform(test['Sex'])

train['Embarked'] = le.fit_transform(train['Embarked'])
test['Embarked'] = le.transform(test['Embarked'])

In [12]:
X = train.drop('Survived', axis=1)
y = train['Survived']

In [13]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)

In [14]:
from sklearn.model_selection import KFold, cross_val_score
def evaluate_kfold_regression(k_list):
    for k in k_list:
        kf = KFold(n_splits=k, shuffle=True, random_state=42)

        # Menghitung Neg Mean Squared Error untuk mendapatkan RMSE
        mse_scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')
        rmse_scores = np.sqrt(-mse_scores)

        print(f"\n=== K-Fold K={k} ===")
        print(f"RMSE tiap fold: {rmse_scores}")
        print(f"RMSE rata-rata: {np.mean(rmse_scores):.4f}")

# Eksekusi K-Fold sesuai tugas
evaluate_kfold_regression([5, 7, 10])


=== K-Fold K=5 ===
RMSE tiap fold: [0.43582581 0.46808239 0.38946809 0.47404546 0.46808239]
RMSE rata-rata: 0.4471

=== K-Fold K=7 ===
RMSE tiap fold: [0.42389562 0.44194174 0.4940592  0.36586646 0.46954493 0.4940592
 0.4778561 ]
RMSE rata-rata: 0.4525

=== K-Fold K=10 ===
RMSE tiap fold: [0.39440532 0.47404546 0.42399915 0.47404546 0.43704832 0.41053541
 0.46204236 0.50835712 0.51929079 0.41053541]
RMSE rata-rata: 0.4514


In [18]:
model.fit(X, y)
# Prediksi berupa angka kontinu
raw_predictions = model.predict(test)

# Thresholding: Mengubah hasil regresi menjadi klasifikasi (0 atau 1)
y_pred = [1 if x >= 0.5 else 0 for x in raw_predictions]
y_true = submit_df['Survived']

In [20]:
from sklearn.metrics import mean_squared_error

print("\n=== Analisis Kinerja Akhir (Linear Regression to Classification) ===")
print(f"RMSE pada Test Data: {np.sqrt(mean_squared_error(y_true, raw_predictions)):.4f}")
print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision : {precision_score(y_true, y_pred):.4f}")
print(f"Recall    : {recall_score(y_true, y_pred):.4f}")
print(f"F1 Score  : {f1_score(y_true, y_pred):.4f}")


=== Analisis Kinerja Akhir (Linear Regression to Classification) ===
RMSE pada Test Data: 0.2396
Accuracy  : 0.9426
Precision : 0.9156
Recall    : 0.9276
F1 Score  : 0.9216
